# CS688 Homework 2

## 1. Goal

Sentiment and theme analysis on 20 English news titles about `ai`, `invest`, and/or `compute`.

- Title text only (not the article body). Each title has at least one search keyword.
- Classical score: AFINN lexicon.
- Modern score: pretrained DistilBERT. I did not train Word2Vec, FastText, or GloVe.
- One manual one-word theme per title. Themes are not search keywords and are not algorithm output.
- Excel file with four columns: Text, Classical Sentiment Score, Modern Sentiment Score, Theme.

## 2. Setup

Run:

```
pip3 install pandas openpyxl afinn transformers torch
```


In [ ]:
import pandas as pd
from afinn import Afinn
from transformers import pipeline

/Users/chrisdavis/personal/cs688_old/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data collection

Same keywords as Homework 1: `ai`, `invest`, `compute`. I manually picked 20 English Bing News titles from `search_results_titles.csv` (20 September 2026). This notebook does not scrape. Only the title is scored.

URLs and outlet names were not kept from the search cards, so they are not invented here and they are not in the Excel file. Several titles contain more than one keyword.


In [3]:
# 20 titles picked by hand from Bing News (20 September 2026).
# Themes were assigned by reading each title.

titles = [
    "Bridgewater's Greg Jensen wants AI compute giants treated like too-big-to-fail banks",
    "SK Hynix Launches Silicon Valley Venture Arm To Invest Across AI Computing, Data Centers And Optical Interconnects",
    "Fastly Highlights Accelerating Growth, Record Margins and AI-Driven Compute Demand",
    "AI compute crunch pays off for CoreWeave as capacity fetches higher prices",
    "Follow Anthropic's Actions, Not It's Rhetoric: 5 AI Stocks to Benefit From Anthropic's $517 Billion in Compute Commitments",
    "Nebius Group (NBIS) Raises AI Cloud Prices As Compute Demand Tests Customer Costs",
    "The AI compute gap: Enterprises are buying infrastructure faster than they can measure what it costs",
    "OpenAI Taps Firmus for Two Malaysia AI Data Centers as APAC Compute Race Accelerates",
    "RUM's $13.7B compute win contradicts AI safety concerns",
    "Jane Street bets $19B on AI compute: Crusoe closes $3B round as finance eclipses labs",
    "Finding where the power is to fuel AI compute",
    "Blackstone plans to invest tens of billions in Google AI chips",
    "Alibaba Quarterly Profit Drops 75% as AI Investment Spending Grows",
    "If an AI bubble burst is coming, history says this 1 investing move is crucial right now",
    "Australia Lags Behind UK and Canada on Sovereign AI Capability as Funding and Compute Gaps Risk Foreign Reliance",
    "Nvidia to Invest $1.5 Billion in SB Energy, Supply AI Compute for Ohio Data Center",
    "SpaceX Wants to Deploy Millions of AI Compute Satellites in Space. Here's How It Plans to Get There.",
    "How Do You Invest In The AI Value Chain?",
    "Anthropic secures $517B in AI compute deals with Amazon, Google over 10 years",
    "US-Europe Investment Gap Widens Due to AI Surge",
]

themes = [
    "Regulation",
    "Expansion",
    "Demand",
    "Pricing",
    "Rhetoric",
    "Pricing",
    "Measurement",
    "Expansion",
    "Safety",
    "Finance",
    "Energy",
    "Chips",
    "Profit",
    "Bubble",
    "Sovereignty",
    "Energy",
    "Satellites",
    "Strategy",
    "Deals",
    "Geography",
]

df = pd.DataFrame()
df["text"] = titles
df["theme"] = themes

print("Selected titles:", len(df))
df


Selected titles: 20
matched_keyword
compute    13
invest      7



,text,publisher,date,matched_keyword,theme
0,Bridgewater's Greg Jensen wants AI compute gia...,Bing News,2026-09-20,compute,Regulation
1,SK Hynix Launches Silicon Valley Venture Arm T...,Bing News,2026-09-20,invest,Expansion
2,"Fastly Highlights Accelerating Growth, Record ...",Bing News,2026-09-20,compute,Demand
3,AI compute crunch pays off for CoreWeave as ca...,Bing News,2026-09-20,compute,Pricing
4,"Follow Anthropic's Actions, Not It's Rhetoric:...",Bing News,2026-09-20,compute,Rhetoric
5,Nebius Group (NBIS) Raises AI Cloud Prices As ...,Bing News,2026-09-20,compute,Pricing
6,The AI compute gap: Enterprises are buying inf...,Bing News,2026-09-20,compute,Measurement
7,OpenAI Taps Firmus for Two Malaysia AI Data Ce...,Bing News,2026-09-20,compute,Expansion
8,RUM's $13.7B compute win contradicts AI safety...,Bing News,2026-09-20,compute,Safety
9,Jane Street bets $19B on AI compute: Crusoe cl...,Bing News,2026-09-20,compute,Finance


## Theme analysis

I assigned themes by reading each title. No keyword extraction, clustering, or LLM.

- One word per title.
- The theme is the meaning of the title, not the search keyword. I did not use `AI`, `Invest`, or `Compute`.
- Repeats are allowed. `Pricing` and `Energy` each appear twice.

This is subjective. Someone else might tag Alibaba as `Spending` instead of `Profit`, or Bridgewater as `Banks` instead of `Regulation`.


## Classical sentiment: AFINN

AFINN gives each known word a fixed score from about -5 to +5. The title score is the **sum**. Unknown words add 0, so I can list which words produced the score. It misses context, negation, and sarcasm.

I treat `> 0` as positive, `< 0` as negative, and `= 0` as neutral. Zero often means the title is factual, or AFINN did not recognize the important words.


In [4]:
afinn = Afinn()

classical_scores = []
classical_labels = []
classical_words = []

for title in df["text"]:
    score = afinn.score(title)
    classical_scores.append(score)

    if score > 0:
        label = "positive"
    elif score < 0:
        label = "negative"
    else:
        label = "neutral"
    classical_labels.append(label)

    # list the AFINN words so the score is easy to check
    hits = []
    words = title.lower().replace("-", " ").split()
    for word in words:
        word = word.strip(".,:'?$()%")
        word_score = afinn.score(word)
        if word_score != 0:
            if word_score > 0:
                hits.append(word + " (+" + str(int(word_score)) + ")")
            else:
                hits.append(word + " (" + str(int(word_score)) + ")")

    if len(hits) == 0:
        classical_words.append("(no lexicon hits)")
    else:
        classical_words.append(", ".join(hits))

df["classical_score"] = classical_scores
df["classical_label"] = classical_labels
df["classical_words"] = classical_words

df[["text", "classical_score", "classical_label", "classical_words"]]


,text,classical_score,classical_label,classical_words
0,Bridgewater's Greg Jensen wants AI compute gia...,1.0,positive,"like (+2), big (+1), fail (-2)"
1,SK Hynix Launches Silicon Valley Venture Arm T...,0.0,neutral,(no lexicon hits)
2,"Fastly Highlights Accelerating Growth, Record ...",1.0,positive,"growth (+2), demand (-1)"
3,AI compute crunch pays off for CoreWeave as ca...,0.0,neutral,(no lexicon hits)
4,"Follow Anthropic's Actions, Not It's Rhetoric:...",2.0,positive,benefit (+2)
5,Nebius Group (NBIS) Raises AI Cloud Prices As ...,-1.0,negative,demand (-1)
6,The AI compute gap: Enterprises are buying inf...,0.0,neutral,(no lexicon hits)
7,OpenAI Taps Firmus for Two Malaysia AI Data Ce...,0.0,neutral,(no lexicon hits)
8,RUM's $13.7B compute win contradicts AI safety...,5.0,positive,"win (+4), safety (+1)"
9,Jane Street bets $19B on AI compute: Crusoe cl...,0.0,neutral,(no lexicon hits)


## Modern sentiment: DistilBERT

I used Hugging Face `distilbert-base-uncased-finetuned-sst-2-english`. It is a pretrained classifier. I did not train Word2Vec, FastText, or GloVe on these 20 titles.

The model was trained on SST-2 movie-review phrases. It returns `POSITIVE` or `NEGATIVE` plus a confidence. It can use context (for example *profit drops*) that AFINN misses, but:

- SST-2 is binary, so there is no `NEUTRAL` class.
- News titles are a different domain than movie reviews.
- High confidence is not the same as being right.

The modern score is stored as `LABEL (confidence)`, for example `NEGATIVE (0.996)`.


In [5]:
clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

modern_labels = []
modern_scores = []

for title in df["text"]:
    result = clf(title)[0]
    label = result["label"]
    conf = round(result["score"], 3)
    modern_labels.append(label)
    modern_scores.append(label + " (" + str(conf) + ")")

df["modern_label"] = modern_labels
df["modern_score"] = modern_scores

df[["text", "classical_score", "modern_score", "theme"]]



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9410.96it/s]

,text,classical_score,modern_score,theme
0,Bridgewater's Greg Jensen wants AI compute gia...,1.0,NEGATIVE (0.999),Regulation
1,SK Hynix Launches Silicon Valley Venture Arm T...,0.0,POSITIVE (0.997),Expansion
2,"Fastly Highlights Accelerating Growth, Record ...",1.0,POSITIVE (1.000),Demand
3,AI compute crunch pays off for CoreWeave as ca...,0.0,NEGATIVE (0.906),Pricing
4,"Follow Anthropic's Actions, Not It's Rhetoric:...",2.0,NEGATIVE (0.987),Rhetoric
5,Nebius Group (NBIS) Raises AI Cloud Prices As ...,-1.0,NEGATIVE (0.948),Pricing
6,The AI compute gap: Enterprises are buying inf...,0.0,NEGATIVE (0.999),Measurement
7,OpenAI Taps Firmus for Two Malaysia AI Data Ce...,0.0,NEGATIVE (0.776),Expansion
8,RUM's $13.7B compute win contradicts AI safety...,5.0,NEGATIVE (0.989),Safety
9,Jane Street bets $19B on AI compute: Crusoe cl...,0.0,NEGATIVE (0.995),Finance


## Excel Table Generation

Four columns only: Text, Classical Sentiment Score, Modern Sentiment Score, Theme. AFINN word hits stay in this notebook.


In [6]:
excel_df = pd.DataFrame()
excel_df["Text"] = df["text"]
excel_df["Classical Sentiment Score"] = df["classical_score"]
excel_df["Modern Sentiment Score"] = df["modern_score"]
excel_df["Theme"] = df["theme"]

excel_df.to_excel("cs688_hw2_davis_chris.xlsx", index=False)
print("Wrote cs688_hw2_davis_chris.xlsx")
excel_df


Wrote cs688_hw2_davis_chris.xlsx with shape (20, 4)
Columns: ['Text', 'Classical Sentiment Score', 'Modern Sentiment Score', 'Theme']


,Text,Classical Sentiment Score,Modern Sentiment Score,Theme
0,Bridgewater's Greg Jensen wants AI compute gia...,1.0,NEGATIVE (0.999),Regulation
1,SK Hynix Launches Silicon Valley Venture Arm T...,0.0,POSITIVE (0.997),Expansion
2,"Fastly Highlights Accelerating Growth, Record ...",1.0,POSITIVE (1.000),Demand
3,AI compute crunch pays off for CoreWeave as ca...,0.0,NEGATIVE (0.906),Pricing
4,"Follow Anthropic's Actions, Not It's Rhetoric:...",2.0,NEGATIVE (0.987),Rhetoric
5,Nebius Group (NBIS) Raises AI Cloud Prices As ...,-1.0,NEGATIVE (0.948),Pricing
6,The AI compute gap: Enterprises are buying inf...,0.0,NEGATIVE (0.999),Measurement
7,OpenAI Taps Firmus for Two Malaysia AI Data Ce...,0.0,NEGATIVE (0.776),Expansion
8,RUM's $13.7B compute win contradicts AI safety...,5.0,NEGATIVE (0.989),Safety
9,Jane Street bets $19B on AI compute: Crusoe cl...,0.0,NEGATIVE (0.995),Finance


## Compare the two methods

AFINN can be neutral. DistilBERT cannot, so it still labels factual titles. I compare polarity when AFINN is non-zero, and note titles where AFINN is 0.


In [7]:
comparisons = []
for i in range(len(df)):
    c_label = df["classical_label"][i]
    m_label = df["modern_label"][i]
    if c_label == "neutral":
        comparisons.append("AFINN neutral / DistilBERT forced polarity")
    elif c_label == "positive" and m_label == "POSITIVE":
        comparisons.append("agree (positive)")
    elif c_label == "negative" and m_label == "NEGATIVE":
        comparisons.append("agree (negative)")
    else:
        comparisons.append("disagree")

df["comparison"] = comparisons

print("AFINN label counts:")
print(df["classical_label"].value_counts())
print()
print("DistilBERT label counts:")
print(df["modern_label"].value_counts())

df[["text", "classical_score", "classical_words", "modern_score", "theme", "comparison"]]


comparison
AFINN neutral / DistilBERT forced polarity    12
disagree                                       5
agree (negative)                               2
agree (positive)                               1

AFINN label counts:
classical_label
neutral     12
positive     6
negative     2

DistilBERT label counts:
modern_label
NEGATIVE    17
POSITIVE     3


,text,classical_score,classical_words,modern_score,theme,comparison
0,Bridgewater's Greg Jensen wants AI compute gia...,1.0,"like (+2), big (+1), fail (-2)",NEGATIVE (0.999),Regulation,disagree
1,SK Hynix Launches Silicon Valley Venture Arm T...,0.0,(no lexicon hits),POSITIVE (0.997),Expansion,AFINN neutral / DistilBERT forced polarity
3,AI compute crunch pays off for CoreWeave as ca...,0.0,(no lexicon hits),NEGATIVE (0.906),Pricing,AFINN neutral / DistilBERT forced polarity
4,"Follow Anthropic's Actions, Not It's Rhetoric:...",2.0,benefit (+2),NEGATIVE (0.987),Rhetoric,disagree
6,The AI compute gap: Enterprises are buying inf...,0.0,(no lexicon hits),NEGATIVE (0.999),Measurement,AFINN neutral / DistilBERT forced polarity
7,OpenAI Taps Firmus for Two Malaysia AI Data Ce...,0.0,(no lexicon hits),NEGATIVE (0.776),Expansion,AFINN neutral / DistilBERT forced polarity
8,RUM's $13.7B compute win contradicts AI safety...,5.0,"win (+4), safety (+1)",NEGATIVE (0.989),Safety,disagree
9,Jane Street bets $19B on AI compute: Crusoe cl...,0.0,(no lexicon hits),NEGATIVE (0.995),Finance,AFINN neutral / DistilBERT forced polarity
10,Finding where the power is to fuel AI compute,0.0,(no lexicon hits),NEGATIVE (0.997),Energy,AFINN neutral / DistilBERT forced polarity
11,Blackstone plans to invest tens of billions in...,0.0,(no lexicon hits),NEGATIVE (0.949),Chips,AFINN neutral / DistilBERT forced polarity


## Our Findings

Many of these titles are facts, not opinions. AFINN is easy to inspect but scores words one at a time. DistilBERT uses context, but it has no `NEUTRAL` class and is not ground truth.

- *Alibaba profit drops 75%*: AFINN = +2 from *profit*. DistilBERT is `NEGATIVE`. AFINN misses *drops*.
- *Compute win contradicts safety*: AFINN = +5 from *win* and *safety*. DistilBERT is `NEGATIVE`. AFINN misses *contradicts*.
- *Too-big-to-fail banks*: AFINN = +1 after *like*, *big*, and *fail* cancel out. DistilBERT is `NEGATIVE` and better captures the warning.
- *AI bubble burst* and *How do you invest...*: AFINN = 0. DistilBERT still picks a side with high confidence. A warning and a question are closer to neutral.

The two scores are complementary. DistilBERT sometimes wins on context. It also over-labels factual and question titles. Themes were assigned by hand and do not repeat the search keywords.
